# Exercise 1 - SVR (Support Vector Regressor) on the housing dataset

In [12]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np

from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import expon, reciprocal
from io import BytesIO
from urllib.request import urlopen
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor

In [7]:
url = "https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv"
housing = pd.read_csv(BytesIO(urlopen(url).read()))
housing.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [8]:
train_set, test_set = train_test_split(housing, test_size=0.2, random_state=42)
X_train = train_set.drop("median_house_value", axis=1)
y_train = train_set["median_house_value"].copy()
num_attribs = X_train.select_dtypes(include=np.number).columns
cat_attribs = ["ocean_proximity"]
num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
preprocessing = ColumnTransformer([("num", num_pipeline, num_attribs), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_attribs)])

In [9]:
X_train_small, y_train_small = X_train[:5000], y_train[:5000]
svr_pipeline = Pipeline([("preprocessing", preprocessing), ("svr", SVR())])

param_grid = [{"svr__kernel": ["linear"], "svr__C": [0.1, 1, 10, 100]},
    {"svr__kernel": ["rbf"], "svr__C": [0.1, 1, 10, 100], "svr__gamma": [0.01, 0.1, 1, 10]}]

svr_search = GridSearchCV(svr_pipeline, param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
svr_search.fit(X_train_small, y_train_small)

print("Best parameters:", svr_search.best_params_)
print("Best CV RMSE:", -svr_search.best_score_)

Best parameters: {'svr__C': 100, 'svr__kernel': 'linear'}
Best CV RMSE: 76252.24370398902


# Exercise 2 - RandomisedSearchCV on SVR

In [10]:
param_distribs = [{"svr__kernel": ["linear"], "svr__C": expon(scale=10)},
    {"svr__kernel": ["rbf"], "svr__C": reciprocal(0.1, 100), "svr__gamma": expon(scale=0.1)}]

svr_random = RandomizedSearchCV(svr_pipeline, param_distributions=param_distribs, n_iter=30, cv=3, scoring="neg_root_mean_squared_error",
                                n_jobs=-1, random_state=42)
svr_random.fit(X_train_small, y_train_small)

print("Best parameters:", svr_random.best_params_)
print("Best CV RMSE:", -svr_random.best_score_)

Best parameters: {'svr__C': np.float64(33.706303424611725), 'svr__kernel': 'linear'}
Best CV RMSE: 87046.88800639358


# Exercise 3 - SelectFromModel transformer on SVR

In [14]:
feature_selector = SelectFromModel(RandomForestRegressor(n_estimators=100, random_state=42), threshold="median")
svr_selected = Pipeline([("preprocessing", preprocessing), ("feature_selection", feature_selector),
                         ("svr", SVR(kernel="rbf", C=svr_random.best_params_.get("svr__C", 1),
                                     gamma=svr_random.best_params_.get("svr__gamma", "scale")))])

svr_selected.fit(X_train_small, y_train_small)
pred = svr_selected.predict(X_train_small)
print("Training RMSE:", np.sqrt(mean_squared_error(y_train_small, pred)))

Training RMSE: 115598.31208801521


- SelectFromModel is a trandformer which selects import features from the dtaaset based on the weights of a previously trained estimator (SVR).
- Here, the randomised search performs wose than the Gridsearch possibly because it could not find the optimal one among the sampled ones that Grid search could find.